# Skip-VAE RL Strong Experiment

Run the setup and debug cells first. Then run the full experiment cells. The final cells display figures and produce the result table.

In [ ]:
!git clone https://github.com/mehddii/skip-vae-rl.git
%cd skip-vae-rl
!git pull
!pip install -q uv
!uv sync --extra notebook

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Debug Pipeline

These runs are intentionally small. They check that VAE training, raw PPO, and pretrained-encoder PPO all execute.

In [ ]:
!uv run python -m skip_vae_rl.train_vae --config configs/debug_vae.yaml
!uv run python -m skip_vae_rl.train_rl --config configs/debug_rl_raw.yaml
!uv run python -m skip_vae_rl.train_rl --config configs/debug_rl_skip_vae.yaml

## Representation Training

This trains the normal VAE, the naive Skip-VAE, and the regularized Skip-VAE. The regularized version uses skip dropout and skip scaling to reduce latent bypassing.

In [ ]:
!uv run python -m skip_vae_rl.train_vae --config configs/vae_minigrid.yaml
!uv run python -m skip_vae_rl.train_vae --config configs/skip_vae_minigrid.yaml
!uv run python -m skip_vae_rl.train_vae --config configs/skip_vae_regularized_minigrid.yaml

## RL Experiments

This matrix compares raw pixels, frozen encoders, and fine-tuned encoders. Evaluation uses multiple seeds.

In [ ]:
!uv run python -m skip_vae_rl.train_rl --config configs/ppo_raw.yaml
!uv run python -m skip_vae_rl.train_rl --config configs/ppo_vae_frozen.yaml
!uv run python -m skip_vae_rl.train_rl --config configs/ppo_vae_finetune.yaml
!uv run python -m skip_vae_rl.train_rl --config configs/ppo_skip_vae_frozen.yaml
!uv run python -m skip_vae_rl.train_rl --config configs/ppo_skip_vae_finetune.yaml
!uv run python -m skip_vae_rl.train_rl --config configs/ppo_skip_vae_regularized_frozen.yaml
!uv run python -m skip_vae_rl.train_rl --config configs/ppo_skip_vae_regularized_finetune.yaml

## Figures And Tables

In [ ]:
!uv run python -m skip_vae_rl.visualize_latent --config configs/latent_vae.yaml
!uv run python -m skip_vae_rl.visualize_latent --config configs/latent_skip_vae.yaml
!uv run python -m skip_vae_rl.visualize_latent --config configs/latent_skip_vae_regularized.yaml
!uv run python -m skip_vae_rl.aggregate_results

In [ ]:
from pathlib import Path
from IPython.display import Image, display

for path in sorted(Path("reports/figures").glob("*.png")):
    print(path)
    display(Image(filename=str(path)))

In [ ]:
import pandas as pd

pd.read_csv("reports/results_summary.csv")

In [ ]:
!zip -r skip_vae_rl_results.zip runs reports